In [1]:
from common.mistral import call_mistral, MistralCallConfig
from common.logger import JUPYTER_LOGGER as logger
from common.paths import get_sber_gitignore_data_dpath

import json
from typing import *
import pandas as pd

In [19]:
features_dpath = get_sber_gitignore_data_dpath() / "output"
features_fpaths: list = sorted(features_dpath.glob("*.csv"))
all_features = []
for fpath in features_fpaths:
    df = pd.read_csv(fpath)
    all_features.append(df)
features_df = pd.concat(all_features, ignore_index=True)
features_df = features_df.dropna(subset=["query", "ground_truth"])
logger.info(f"Размер итогового датафрейма признаков: {features_df.shape}")

2026-03-31 02:06:51,147 - jupyter-notebooks - INFO - [JUPYTER 📓] Размер итогового датафрейма признаков: (776, 1620)


In [3]:
judge_prompt_template = """
Ты — судья, оценивающий точность ответа модели.
Галлюцинация — это когда модель генерирует информацию, не соответствующую фактам и недостоверную.

Вопрос: {query}
Правильный ответ (достоверный эталон): {ground_truth}
Ответ модели: {model_answer}

Определи, является ли ответ модели галлюцинацией.
Выбери строго один вариант и ничего больше: "галлюцинация" или "не галлюцинация".
ТОЛЬКО эти два слова, без кавычек и без дополнительных пояснений.
"""
prompts = [judge_prompt_template.format(query=row["query"], ground_truth=row["ground_truth"], model_answer=row["model_answer"]) for _, row in features_df.iterrows()]
features_df["judge_prompt"] = prompts

C:\Users\User\AppData\Local\Temp\ipykernel_15804\3639616577.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features_df["judge_prompt"] = prompts


In [4]:
from tqdm.auto import tqdm

mistral_cfg = MistralCallConfig(
    models_list=["mistral-small-2603"],
)

judge_system_prompt = (
    "Ты строгий факт-чекер и судья качества ответов. "
    "Проверяй соответствие ответа модели эталонному ответу. "
    "Если доступен внешний инструмент поиска, используй его перед финальным вердиктом. "
    "Отвечай только в требуемом JSON-формате."
)

# Для детерминированной и воспроизводимой оценки.
judge_generation_kwargs = {
    "temperature": 0.1,
    "top_p": 0.9,
    "presence_penalty": 0.0,
    "frequency_penalty": 0.0,
    "reasoning_effort": "high",
    "response_format": {"type": "json_object"},
}

# При наличии интеграции можно передать tools (например, web-search/knowledge-base).
# Если tools пустой, вызов остаётся совместимым с текущим пайплайном.
judge_tools: list[dict[str, Any]] = []
judge_tool_choice: str = "required" if judge_tools else "auto"

output_fpath = get_sber_gitignore_data_dpath() / "judge_scores.csv"
scores = []
queries = []
gt = []
answers = []

def save_scores():
    scores_df = {
        "query": queries,
        "ground_truth": gt,
        "model_answer": answers,
        "judge_score": scores,
    }
    pd.DataFrame(scores_df).to_csv(output_fpath, index=False)


def _normalize_judge_score(raw_response: str) -> str:
    text = raw_response.strip().lower()

    # Предпочитаем структурированный ответ из response_format=json_object.
    try:
        parsed: Any = json.loads(text)
        if isinstance(parsed, dict):
            score_value: Any = parsed.get("judge_score")
            if isinstance(score_value, str):
                normalized = score_value.strip().lower()
                if normalized in {"галлюцинация", "не галлюцинация"}:
                    return normalized
    except json.JSONDecodeError:
        pass

    if "не галлюцинация" in text:
        return "не галлюцинация"
    if "галлюцинация" in text:
        return "галлюцинация"
    return "неизвестно"


def _build_call_kwargs(prompt: str) -> dict[str, Any]:
    messages: list[dict[str, str]] = [
        {"role": "system", "content": judge_system_prompt},
        {"role": "user", "content": prompt},
    ]
    call_kwargs: dict[str, Any] = {
        "messages": messages,
        **judge_generation_kwargs,
    }
    if judge_tools:
        call_kwargs["tools"] = judge_tools
        call_kwargs["tool_choice"] = judge_tool_choice
    return call_kwargs

def make_call(raw):
    prompt = raw.judge_prompt
    call_kwargs = _build_call_kwargs(prompt)
    response = call_mistral(mistral_cfg, **call_kwargs)
    score = _normalize_judge_score(response)
    scores.append(score)
    queries.append(raw.query)
    gt.append(raw.ground_truth)
    answers.append(raw.model_answer)
    save_scores()

# test call
import random
test_raw = features_df.iloc[random.randint(0, features_df.shape[0] - 1)]
make_call(test_raw)
test_score = scores[-1]
logger.info(f"Тестовый вызов завершился. Промпт:\n{test_raw.judge_prompt}\nОценка судьи: {test_score}")

2026-03-31 01:47:51,102 - mistral-call - INFO - [MISTRAL 🇫🇷] call_mistral: начало вызова API
2026-03-31 01:47:51,104 - mistral-call - DEBUG - [MISTRAL 🇫🇷] call_mistral: параметры - ['messages', 'temperature', 'top_p', 'presence_penalty', 'frequency_penalty', 'reasoning_effort', 'response_format']
2026-03-31 01:47:51,106 - mistral-call - INFO - [MISTRAL 🇫🇷] call_mistral: попытка 1/1 с моделью mistral-small-2603, ключ 1/13
2026-03-31 01:47:51,497 - mistral-call - DEBUG - [MISTRAL 🇫🇷] safe_call: вызов функции с timeout=240
2026-03-31 01:47:51,498 - mistral-call - DEBUG - [MISTRAL 🇫🇷] safe_call: попытка 1
2026-03-31 01:47:51,500 - mistral-call - DEBUG - [MISTRAL 🇫🇷] sub_call: попытка вызова модели mistral-small-2603
2026-03-31 01:47:51,792 - mistral-call - ERROR - [MISTRAL 🇫🇷] safe_call: ошибка на попытке 1: API error occurred: ...
Traceback (most recent call last):
  File "C:\Users\User\Desktop\dirs\Dev\hack-mfti\common\mistral.py", line 164, in safe_call
    result = future.result(timeou

In [5]:
pbar = tqdm(total=len(features_df), desc="Оценка ответов судьей")
import logging
logging.disable(logging.DEBUG)
logging.disable(logging.INFO)
for _, row in features_df.iterrows():
    make_call(row)
    pbar.update(1)
pbar.close()
logging.disable(logging.NOTSET)
logger.info(f"Оценка всех ответов завершена. Результаты сохранены в {output_fpath}")

Оценка ответов судьей:   0%|          | 0/776 [00:00<?, ?it/s]

2026-03-31 01:48:47,307 - mistral-call - ERROR - [MISTRAL 🇫🇷] safe_call: ошибка на попытке 1: API error occurred: ...
Traceback (most recent call last):
  File "C:\Users\User\Desktop\dirs\Dev\hack-mfti\common\mistral.py", line 164, in safe_call
    result = future.result(timeout=timeout)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\User\anaconda3\envs\hack-mfti\Lib\concurrent\futures\_base.py", line 456, in result
    return self.__get_result()
           ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\User\anaconda3\envs\hack-mfti\Lib\concurrent\futures\_base.py", line 401, in __get_result
    raise self._exception
  File "C:\Users\User\anaconda3\envs\hack-mfti\Lib\concurrent\futures\thread.py", line 59, in run
    result = self.fn(*self.args, **self.kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\User\Desktop\dirs\Dev\hack-mfti\common\mistral.py", line 192, in sub_call
    response: Any = client.chat.complete(model=model_name, **sub_kwargs)
             

In [22]:
final_scores_df = pd.DataFrame(
    {
        "query": queries,
        "ground_truth": gt,
        "model_answer": answers,
        "judge_score": scores,
    }
)

# Нормализуем текстовые ключи для надежного merge
for col in ["query", "model_answer", "judge_score"]:
    final_scores_df[col] = final_scores_df[col].astype("string").str.strip()

# Оставляем только валидные метки судьи
valid_labels = {"галлюцинация", "не галлюцинация"}
final_scores_df = final_scores_df[final_scores_df["judge_score"].isin(valid_labels)].copy()

# Удаляем дубликаты по ключу (берем последнюю оценку)
final_scores_df = final_scores_df.drop_duplicates(
    subset=["query", "model_answer"],
    keep="last",
)

# Преобразуем в boolean: True = галлюцинация
final_scores_df["judge_score"] = final_scores_df["judge_score"].eq("галлюцинация")

score_counts = final_scores_df["judge_score"].value_counts(dropna=False)
logger.info("Распределение оценок судьи:")
for score, count in score_counts.items():
    logger.info(f"{score}: {count} ({count / max(len(final_scores_df), 1) * 100:.1f}%)")

# Нормализуем ключи и в features_df тем же способом
for col in ["query", "model_answer"]:
    features_df[col] = features_df[col].astype("string").str.strip()

# Если столбец уже есть после прошлых запусков — удаляем перед merge
if "judge_score" in features_df.columns:
    features_df = features_df.drop(columns=["judge_score"])

features_df = features_df.merge(
    final_scores_df[["query", "model_answer", "judge_score"]],
    on=["query", "model_answer"],
    how="left",
)

matched = int(features_df["judge_score"].notna().sum())
total = len(features_df)
logger.info(f"После merge: {matched} из {total} строк имеют оценку судьи ({matched / max(total, 1) * 100:.1f}%)")

output_path = get_sber_gitignore_data_dpath() / "features_with_judge_scores.csv"
features_df.to_csv(output_path, index=False)
logger.info(f"Итоговый датафрейм сохранён: {output_path}")


2026-03-31 02:09:15,473 - jupyter-notebooks - INFO - [JUPYTER 📓] Распределение оценок судьи:
2026-03-31 02:09:15,475 - jupyter-notebooks - INFO - [JUPYTER 📓] True: 571 (76.4%)
2026-03-31 02:09:15,477 - jupyter-notebooks - INFO - [JUPYTER 📓] False: 176 (23.6%)
2026-03-31 02:09:15,494 - jupyter-notebooks - INFO - [JUPYTER 📓] После merge: 456 из 462 строк имеют оценку судьи (98.7%)
2026-03-31 02:09:16,382 - jupyter-notebooks - INFO - [JUPYTER 📓] Итоговый датафрейм сохранён: C:\Users\User\Desktop\dirs\Dev\hack-mfti\sber\data\gitignore\features_with_judge_scores.csv
